# Statistical significance on the NASA held-out test set

This notebook recomputes the manuscript comparison of TE-Q-Transformer with iTransformer and QNN-GRU on the 187 held-out NASA cycles (B0018: 132, B0032: 39, B0053 test portion: 16).

The primary endpoint is the paired cycle-level squared-error difference

$$d_i = (y_i - \hat y_{\mathrm{TEQ},i})^2 - (y_i - \hat y_{\mathrm{base},i})^2.$$

Negative values favor TE-Q-Transformer. Tests are a paired *t*-test and a Wilcoxon signed-rank test, with Holm adjustment across the two baselines. A moving-block bootstrap ($B = 2000$, block length $L = 5$, within each cell, seed 42) supplies a dependence-aware interval. Cycle-level tests treat cycles as independent, so the bootstrap is the secondary check for serial dependence. These two baselines are the ones in the manuscript statistical table. The tests do not establish superiority over every other baseline.


In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "nasa_manuscript" / "nasa_heldout_predictions.csv").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from the TE-Q-Transformer repository.")

ROOT = repo_root()
PRED_PATH = ROOT / "data" / "nasa_manuscript" / "nasa_heldout_predictions.csv"
print("Repository root:", ROOT)
print("Prediction file:", PRED_PATH)

from scipy import stats
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
df = pd.read_csv(PRED_PATH)
needed = ["TE-Q-Transformer", "iTransformer", "QNN-GRU"]
missing = [name for name in needed if name not in set(df["model"])]
if missing:
    raise RuntimeError(f"Missing models: {missing}")
print("rows", len(df), "seed", sorted(df["seed"].unique()))


Repository root: E:\TE-Q-Transformer\TE-Q-Transformer
Prediction file: E:\TE-Q-Transformer\TE-Q-Transformer\data\nasa_manuscript\nasa_heldout_predictions.csv


rows 2057 seed [np.int64(42)]


## Data validation


In [2]:
cells = ["B0018", "B0032", "B0053_test"]
expected = {"B0018": 132, "B0032": 39, "B0053_test": 16}
frames = []
for model in ["TE-Q-Transformer", "iTransformer", "QNN-GRU"]:
    part = df[df["model"] == model].copy()
    counts = part.groupby("cell").size().to_dict()
    if counts != expected:
        raise RuntimeError(f"{model} counts {counts} != {expected}")
    frames.append(part.sort_values(["cell", "cycle_id"]).reset_index(drop=True))
teq, itr, qnn = frames
if not np.allclose(teq["true_soh"], itr["true_soh"]) or not np.allclose(teq["true_soh"], qnn["true_soh"]):
    raise RuntimeError("True SOH is not aligned across models.")
if not np.allclose(teq["cycle_id"], itr["cycle_id"]) or not np.array_equal(teq["cell"], itr["cell"]):
    raise RuntimeError("Cycle keys are not aligned.")
y = teq["true_soh"].to_numpy(dtype=float)
pred = {
    "TE-Q-Transformer": teq["pred_soh"].to_numpy(dtype=float),
    "iTransformer": itr["pred_soh"].to_numpy(dtype=float),
    "QNN-GRU": qnn["pred_soh"].to_numpy(dtype=float),
}
labels = teq["cell"].to_numpy()
print("Aligned cycles:", len(y))
print(teq.groupby("cell").size().to_dict())
print("True SOH range:", float(y.min()), float(y.max()))


Aligned cycles: 187
{'B0018': 132, 'B0032': 39, 'B0053_test': 16}
True SOH range: 0.7229370474815369 1.099658727645874


## Paired tests and Holm correction


In [3]:
def holm(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    running = 0.0
    m = len(p_values)
    for rank, index in enumerate(order):
        value = min(1.0, float(p_values[index]) * (m - rank))
        running = max(running, value)
        adjusted[index] = running
    return adjusted

def paired(name, loss_teq, loss_base):
    diff = loss_teq - loss_base
    t_stat, t_p = stats.ttest_rel(loss_teq, loss_base)
    w_stat, w_p = stats.wilcoxon(loss_teq, loss_base, alternative="two-sided", zero_method="wilcox")
    return {"name": name, "diff": diff, "t": float(t_stat), "t_p": float(t_p), "w": float(w_stat), "w_p": float(w_p)}

sq = {name: (y - pred[name]) ** 2 for name in pred}
abs_err = {name: np.abs(y - pred[name]) for name in pred}
primary = [
    paired("iTransformer", sq["TE-Q-Transformer"], sq["iTransformer"]),
    paired("QNN-GRU", sq["TE-Q-Transformer"], sq["QNN-GRU"]),
]
secondary = [
    paired("iTransformer", abs_err["TE-Q-Transformer"], abs_err["iTransformer"]),
    paired("QNN-GRU", abs_err["TE-Q-Transformer"], abs_err["QNN-GRU"]),
]

def show(title, rows):
    t_adj = holm([row["t_p"] for row in rows])
    w_adj = holm([row["w_p"] for row in rows])
    table = pd.DataFrame({
        "Baseline": [row["name"] for row in rows],
        "Mean difference": [float(np.mean(row["diff"])) for row in rows],
        "t": [row["t"] for row in rows],
        "t p raw": [row["t_p"] for row in rows],
        "t p Holm": t_adj,
        "Wilcoxon W": [row["w"] for row in rows],
        "Wilcoxon p raw": [row["w_p"] for row in rows],
        "Wilcoxon p Holm": w_adj,
    })
    print(title)
    print(table.to_string(index=False))
    return table

primary_table = show("Squared error", primary)
secondary_table = show("Absolute error", secondary)


Squared error
    Baseline  Mean difference          t      t p raw     t p Holm  Wilcoxon W  Wilcoxon p raw  Wilcoxon p Holm
iTransformer        -0.002035 -12.270999 9.643984e-26 1.928797e-25       493.0    4.396464e-29     8.792929e-29
     QNN-GRU        -0.000282  -3.121721 2.085135e-03 2.085135e-03      6864.0    9.396166e-03     9.396166e-03
Absolute error
    Baseline  Mean difference          t      t p raw     t p Holm  Wilcoxon W  Wilcoxon p raw  Wilcoxon p Holm
iTransformer        -0.024030 -15.331921 7.786353e-35 1.557271e-34       658.0    5.286331e-28     1.057266e-27
     QNN-GRU        -0.004353  -2.746040 6.624311e-03 6.624311e-03      7148.0    2.682134e-02     2.682134e-02


## Moving-block bootstrap


In [4]:
def block_bootstrap_mean(differences, cell_labels, n_boot=2000, block_len=5, seed=42):
    np.random.seed(seed)
    means = np.empty(n_boot)
    groups = {cell: differences[cell_labels == cell] for cell in pd.unique(cell_labels)}
    for draw in range(n_boot):
        pieces = []
        for values in groups.values():
            n_cycles = len(values)
            length = min(block_len, n_cycles)
            n_blocks = int(math.ceil(n_cycles / length))
            starts = np.random.randint(0, max(1, n_cycles - length + 1), size=n_blocks)
            index = []
            for start in starts:
                index.extend(range(int(start), min(int(start) + length, n_cycles)))
            pieces.append(values[np.asarray(index[:n_cycles])])
        means[draw] = np.concatenate(pieces).mean()
    return np.quantile(means, [0.025, 0.975])

print("B = 2000, L = 5, seed = 42, blocks drawn inside each cell")
for row in primary:
    low, high = block_bootstrap_mean(row["diff"], labels)
    print(f"{row['name']}: mean {np.mean(row['diff']):.6e}, 95% CI [{low:.6e}, {high:.6e}]")


B = 2000, L = 5, seed = 42, blocks drawn inside each cell


iTransformer: mean -2.035265e-03, 95% CI [-2.633475e-03, -1.380385e-03]


QNN-GRU: mean -2.815510e-04, 95% CI [-5.767457e-04, -3.469646e-05]


## Interpretation

The squared-error intervals are the moving-block bootstrap intervals. They address serial dependence within a cell. The paired tests themselves still use cycle-level observations. Only iTransformer and QNN-GRU are tested, because those are the two comparisons reported in the manuscript. A significant pooled result is sample-weighted: B0018 contributes 132 of the 187 cycles. It is not a claim of equal improvement on every cell, and it is not a claim of superiority over the other eight baselines.
